# Python-1, Лекция 10

Лектор: Хайбулин Даниэль

Подготовил материал: Лущ Иван

Итак, сегодня мы поговорим про модули `collections`, `heapq` и `itertools`

### Collections

`collections` — модуль стандартной библиотеки с дополнительными структурами данных: `defaultdict`, `Counter`, `deque`, `namedtuple`, `OrderedDict` и другими. Каждая из них решает задачу, для которой встроенных типов не хватает или с ними получается громоздко.

#### defaultdict

`defaultdict` — словарь, который сам создаёт значение для отсутствующего ключа. При первом обращении к новому ключу он вызывает переданную фабрику (`default_factory`) и кладёт результат в словарь. Проверять наличие ключа или звать `get` больше не нужно — в остальном это обычный `dict`.

In [ ]:
from collections import defaultdict

# фабрика вызывается без аргументов и даёт значение для нового ключа
numbers: defaultdict[int, float] = defaultdict(float)
buckets: defaultdict[int, list[int]] = defaultdict(list)

print(numbers[2], buckets[2])

Если не передавать функцию по умолчанию (или передать `None`), `defaultdict` по сути ведёт себя как обычный `dict`: не создаёт новых элементов автоматически и выбрасывает `KeyError` при обращении к отсутствующему ключу.


Использование defaultdict без указания функции по умолчанию (default_factory=None) не рекомендуется, поскольку такой подход может ввести в заблуждение разработчиков, сопровождающих код.

In [ ]:
dct: dict[int, int] = defaultdict()  # то же самое, что defaultdict(None)

dct[2]

##### Сравнение использования dict и defaultdict

In [ ]:
counter: dict[str, int] = dict()

elements: list[str] = ["apple", "banana", "apple", "orange", "banana", "apple"]

for fruit in elements:
    if fruit not in counter:
        counter[fruit] = 0
    counter[fruit] += 1

print(counter)

с использованием `get`:

In [ ]:
counter: dict[str, int] = dict()

elements: list[str] = ["apple", "banana", "apple", "orange", "banana", "apple"]

for fruit in elements:
    counter[fruit] = counter.get(fruit, 0) + 1

print(counter)

In [ ]:
# Создаем defaultdict с функцией (int) по умолчанию, которая возвращает 0
counter: defaultdict[str, int] = defaultdict(int)
# Можно еще написать так (это больше для понимая того,
# что во внутрь передается объект,
# который мы можем вызвать через круглые скобки), но такой код писать не стоит
# counter = defaultdict(lambda : 0)

elements: list[str] = ["apple", "banana", "apple", "orange", "banana", "apple"]

for fruit in elements:
    counter[fruit] += 1

print(counter)

С помощью `defaultdict` не требуется явно проверять наличие ключа или использовать метод `get()` для установки значения по умолчанию. Это уменьшает количество шаблонного кода и делает логику работы с данными более понятной.

Вторая типичная задача для `defaultdict` — не считать, а **группировать**. Фабрика `list` создаёт пустой список для каждого нового ключа, и остаётся только дописывать в него.

In [ ]:
from collections import defaultdict

words: list[str] = ["кот", "кит", "собака", "слон", "сом", "крот"]

by_letter: defaultdict[str, list[str]] = defaultdict(list)
for word in words:
    by_letter[word[0]].append(word)

for letter, group in by_letter.items():
    print(letter, group)

##### Антипатерны

Проверять наличие ключа обращением по индексу нельзя: такое обращение **создаёт** элемент. Словарь незаметно растёт, а условие всегда истинно.

In [ ]:
# неправильно

dct: defaultdict[int, float] = defaultdict(float)

for i in range(5):
    if dct[i] == 0:
        print("Found")

dct

In [ ]:
# правильно

dct: defaultdict[int, float] = defaultdict(float)

for i in range(5):
    if i in dct:
        print("Found")

dct

Что не так с кодом? Какой будет вывод?

In [ ]:
# неправильно

lst = []
dct: defaultdict[int, list[int]] = defaultdict(lambda: lst)

for i in range(5):
    dct[i].append(1)

dct

In [ ]:
# правильно

dct: defaultdict[int, list[int]] = defaultdict(list)

for i in range(5):
    dct[i].append(1)

dct

#### namedtuple

`namedtuple` — это **функция** из модуля `collections`, позволяющая создавать неизменяемые (**immutable**) объекты-кортежи с именованными полями. Такая структура обеспечивает удобный доступ к полям как по имени, так и по индексу, сохраняя компактность и неизменяемость стандартных кортежей.

In [ ]:
from collections import namedtuple

Point = namedtuple("Point", ["x", "y"])
p1 = Point(2, 3)

print(p1.x)  # 2

# Как правило, при использовании namedtuple обращения к элементам по индексам не применяются,
# поскольку такой подход снижает читаемость и ясность кода.
# Вместо этого рекомендуется использовать доступ к значениям по именованным полям,
# что делает структуру данных более наглядной и облегчает сопровождение программы.
print(p1[1])  # 3

print(p1)  # Point(x=2, y=3)

# Нечитаемый код
result = p1[0] + p1[1]  # Не понятно, что тут x и y
print(result)

Еще раз хочется повториться, что `namedtuple` создает именно неизменяемые объекты

In [ ]:
Point = namedtuple("Point", ["x", "y"])
p = Point(1, 2)
p.x = 5

Рядом стоит `dataclass` из модуля `dataclasses` — тоже способ описать структуру с именованными полями, но с аннотациями типов и возможностью сделать её изменяемой или нет. Декораторы и классы разбираются отдельно; здесь достаточно посмотреть на результат.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Point:
    x: int
    y: int


p2 = Point(2, 3)
print(p2.x)  # 2
print(p2)  # Point(x=2, y=3)

Из-за `frozen=True` не можем менять значения полей:

In [ ]:
p2.x = 3

Что выбрать — зависит от того, нужна ли кортежная семантика. `namedtuple` **остаётся кортежем**: распаковывается, хешируется, сравнивается с обычными кортежами. `dataclass` этого не умеет, зато гибче: изменяемые поля, значения по умолчанию, методы, наследование.

У `namedtuple` есть второй, более современный способ записи — класс, унаследованный от `typing.NamedTuple`. Получается тот же самый кортеж с именованными полями, но поля объявляются с аннотациями типов, и можно добавлять методы и значения по умолчанию.

In [ ]:
from typing import NamedTuple


class Measurement(NamedTuple):
    sensor: str
    value: float
    ok: bool = True


m = Measurement("t1", 36.6)
print(m, m.value, m.ok)

# это по-прежнему кортеж: распаковывается и сравнивается с обычным кортежем
sensor, value, ok = m
print(sensor, m == ("t1", 36.6, True))

#### Counter

`Counter` — это специализированный класс, предназначенный для подсчёта количества вхождений элементов в итерируемых объектах (строках, списках и т.д.). По сути, это словарь, где ключами являются элементы, а значениями — количество их появлений.

In [ ]:
from collections import Counter


lst: list[str] = ["apple", "banana", "apple", "orange", "banana", "apple"]
c: Counter[str] = Counter(lst)
print(c)

подсчёт символов в строке:

In [ ]:
from collections import Counter


text = "abracadabra"
c: Counter[str] = Counter(text)
print(c)

Использование метода `most_common()`:

In [ ]:
from collections import Counter


words: list[str] = ["cat", "dog", "cat", "mouse", "dog", "cat"]
c: Counter[str] = Counter(words)
print(c.most_common(2))

Получение элементов с повторениями:

Счётчики можно складывать и вычитать — как мультимножества. Ради этого `Counter` чаще всего и берут вместо обычного словаря.

* `a + b` — сложить количества;
* `a - b` — вычесть, отрицательные и нулевые значения отбрасываются;
* `a & b` — минимум по каждому ключу, то есть общая часть;
* `a | b` — максимум по каждому ключу.

In [ ]:
from collections import Counter

needed = Counter({"мука": 300, "сахар": 100, "яйцо": 2})
have = Counter({"мука": 500, "сахар": 40})

print("не хватает:", needed - have)
print("общая часть:", needed & have)
print("вместе:     ", needed + have)

In [ ]:
from collections import Counter


c: Counter[str] = Counter(a=2, b=3, c=1)
print(list(c.elements()))

`Counter` — подкласс `dict`, поэтому всё словарное работает: обращение по ключу, `items()`, `keys()`, `values()`, итерация.

#### Deque

`deque` — двусторонняя очередь: добавление и удаление с обоих концов за `O(1)`. У списка вставка в начало стоит `O(n)`, потому что все элементы приходится сдвигать.

* `Stack` — LIFO, последний пришёл — первый вышел (стопка тарелок)
* `Queue` — FIFO, первый пришёл — первый вышел (очередь в банке)
* `deque` — и то, и другое: работает с обоих концов

С помощью deque можно создать двустороннюю очередь из любой последовательности, например из списка строк. После создания можем вывести все элементы очереди.

In [ ]:
from collections import deque


s: list[str] = ["ab", "ab", "bc", "cd", "ab"]
d: deque[str] = deque(s)
print(d)

print("-" * 30)

for elem in d:
    print(elem)

print("-" * 30)

Методы `append()` и `appendleft()` позволяют добавлять элементы в конец и начало очереди соответственно. С помощью .extend() и .extendleft() можно добавить несколько элементов сразу.

In [ ]:
d.append("aa")  # Добавление в конец
d.appendleft("dd")  # Добавление в начало
d.extend(["hh", "pp"])  # Добавление нескольких элементов в конец
d.extendleft(["hh", "pp"])  # Добавление нескольких элементов в начало

print(d)

Методы `pop()` и `popleft()` удаляют и возвращают элементы соответственно с правого и левого конца очереди.

In [ ]:
print(d.pop())  # Удаляет последний элемент
print(d.popleft())  # Удаляет первый элемент
print(d)

Можно обращаться к элементам по индексу, а также использовать методы `count()` для подсчета вхождений и `index()` для поиска индекса первого вхождения.

In [ ]:
print(d[0])  # Первый элемент
print(d[-2])  # Второй элемент с конца
print(d.count("hh"))  # Число вхождений 'hh'

Метод `rotate(n)` сдвигает элементы очереди на `n` позиций вправо (если `n` положительно) или влево (если `n` отрицательно). Метод `reverse()` разворачивает очередь.

In [ ]:
d.rotate(1)  # Сдвиг на 1 вправо
print(d)
d.rotate(-2)  # Сдвиг на 2 влево
print(d)
d.reverse()  # Разворот очереди
print(d)

Можно вставлять элементы по заданному индексу с помощью `insert()`, находить их позицию методом `index()`, а также удалять первое вхождение определенного значения с помощью `remove()`.

У `deque` есть параметр `maxlen`. Очередь с ним не растёт бесконечно: когда она заполнена, добавление с одного конца вытесняет элемент с другого. Это стандартный способ хранить «последние N» чего угодно — без ручной обрезки.

In [ ]:
from collections import deque

last_errors: deque[str] = deque(maxlen=3)

for line in ["ошибка 1", "ошибка 2", "ошибка 3", "ошибка 4", "ошибка 5"]:
    last_errors.append(line)
    print(list(last_errors))

In [ ]:
d.insert(1, "x")  # Вставка элемента на позицию 1
print(d)
print(d.index("x"))  # Индекс элемента 'x'
d.remove("x")  # Удаление первого 'x'
print(d)

#### OrderedDict

`OrderedDict` — словарь, который помнит порядок добавления и умеет им управлять: `move_to_end()` переносит ключ в конец или начало, `popitem(last=...)` снимает элемент с нужного конца.

In [ ]:
from collections import OrderedDict


d: OrderedDict[str, int] = OrderedDict()
d["one"] = 1
d["two"] = 2
d["three"] = 3

for key, value in d.items():
    print(key, value)

Использование методов `move_to_end` и `popitem`:

In [ ]:
d.move_to_end("one")  # Перемещает 'one' в конец
print(list(d.keys()))  # ['two', 'three', 'one']

d.popitem(last=False)  # Убирает первый элемент ('two')
print(list(d.keys()))  # ['three', 'one']

d.popitem()  # Убирает последний элемент ('one'), по умолчанию last=True
print(list(d.keys()))  # ['three']

Начиная с версии `Python 3.7` (официально задокументировано в `Python 3.7`), стандартный `dict` сохраняет порядок добавления элементов точно так же, как это делал `OrderedDict`. Таким образом, во многих ситуациях надобность в использовании `OrderedDict` исчезла, и для большинства задач достаточно обычного словаря.

Когда все ещё полезен `OrderedDict`?

- Если требуются специальные методы (`move_to_end`, `popitem` с параметром). Например, при реализации логики `lru_cache`, очень удобно использовать именно `OrderedDict`. 
- Если важен явный контракт на поддержку порядка в старых версиях `Python (< 3.7)`.


#### heapq

`heapq` — это стандартный модуль в Python, предоставляющий набор функций для работы с кучей (heap), реализованной на базе обычного списка (`list`). Куча всегда хранит на первом месте (`heap[0]`) наименьший элемент. Основным назначением кучи являются эффективные операции получения и удаления минимальных (или максимальных, с некоторыми модификациями) элементов.


- `heapq.heapify(x)` — преобразует список x в кучу за время `O(n)`.
- `heapq.heappush(heap, item)` — добавляет элемент `item` в кучу.
- `heapq.heappop(heap)` — удаляет и возвращает наименьший элемент из кучи.
- `heapq.heappushpop(heap, item)` — вставляет элемент и возвращает наименьший за одну операцию.
- `heapq.heapreplace(heap, item)` — удаляет и возвращает наименьший, одновременно добавляя новый элемент.
- `heapq.nlargest(n, iterable)` — возвращает `n` наибольших элементов из последовательности.
- `heapq.nsmallest(n, iterable)` — возвращает `n` наименьших элементов.


In [ ]:
import heapq

# 1. Преобразование списка в кучу
lst: list[int] = [9, 3, 7, 1, 4]
heapq.heapify(lst)
print(lst)  # [1, 3, 7, 9, 4]

# 2. Добавление элемента
heapq.heappush(lst, 2)
print(lst)  # [1, 3, 2, 9, 4, 7]

# 3. Извлечение минимального значения
min_elem = heapq.heappop(lst)
print(min_elem)  # 1
print(lst)  # [2, 3, 7, 9, 4]

# 4. Одновременное добавление нового элемента и извлечение минимального
print(heapq.heappushpop(lst, 0))  # 0 (т.к. '0' меньше текущего минимума)
print(lst)  # [2, 3, 7, 9, 4]

# 5. Получение наибольших и наименьших элементов
nums = [2, 8, 5, 1, 6, 7]
print(heapq.nlargest(3, nums))  # [8, 7, 6]
print(heapq.nsmallest(2, nums))  # [1, 2]

### Itertools

`itertools` — модуль ленивых функций для работы с итераторами: они не строят промежуточных списков и работают с бесконечными последовательностями. Ниже — те функции, которые встречаются чаще всего; полный перечень в [документации](https://docs.python.org/3/library/itertools.html).

#### chain

In [ ]:
from itertools import chain

`itertools.chain(*iterables)` — это функция, которая принимает несколько итерируемых объектов (например, списки, строки, кортежи и т.д.) и возвращает единый итератор, проходящий по всем элементам этих итерируемых объектов последовательно, как если бы они были объединены в одну длинную последовательность.

In [ ]:
for e in chain(range(3), [1, 2], "lol", [[i] for i in range(4)]):
    print(e, end=" ")

В отличие от простого сложения списков `(a + b + c)`, chain работает с любыми итерируемыми объектами и не создает дополнительных коллекций в памяти.

Есть также вариант `chain.from_iterable(iterable_of_iterables)`, когда все последовательности заранее собраны в одном итерируемом объекте (например, в списке).

In [ ]:
iterables = [[1, 2], ("a", "b"), [3, 4]]
for x in chain.from_iterable(iterables):
    print(x)

#### product

In [ ]:
from itertools import product

`itertools.product(*iterables, repeat=1)` — это функция, возвращающая декартово произведение входных итерируемых последовательностей.

In [ ]:
roles: tuple[str, str, str] = ("bs", "yabs", "bsrank")
modes: tuple[str, str] = ("meta", "stat")

for r in roles:
    for m in modes:
        print(f"{r=}, {m=}")

Использование функции `product` повышает читаемость и ясность кода, позволяя явно выразить перебор всех возможных комбинаций элементов из заданных последовательностей без необходимости использования вложенных циклов:

In [ ]:
for r, m in product(roles, modes):
    print(f"{r=}, {m=}")

In [ ]:
a = product("AB", "12")
print(list(a))

#### islice, count, cycle, repeat

In [ ]:
from itertools import count, cycle, repeat, islice

`itertools.islice(iterable, start, stop[, step])` — срез по итератору. В отличие от `seq[a:b:c]` работает с любым итерируемым объектом, в том числе бесконечным, и ничего не загружает в память целиком.

In [ ]:
sequence = range(10)

print(list(islice(sequence, 2, 5)))  # как sequence[2:5]
print(list(islice(sequence, 2, 8, 2)))  # как sequence[2:8:2]

`itertools.count(start=0, step=1)` — это бесконечный итератор, который генерирует арифметическую последовательность чисел, начиная с значения `start` (по умолчанию `0`) и увеличивая (или уменьшая, если `step` отрицателен) предыдущее значение на величину `step` (по умолчанию `1`).

In [ ]:
i: count[int] = count(10, 2)
print(next(i))
print(next(i))
print(next(i))
print(next(i))

list(islice(count(start=0, step=3), 10))

В отличие от `range`, `itertools.count()`:
- Бесконечный итератор: последовательность не имеет фиксированного конца и может продолжаться бесконечно (пока не остановить явно).
- Принимает любые значения для `start` и `step`, в том числе числа с плавающей запятой (`float`), отрицательные значения, ноль.
- Возвращает объект-итератор.

In [ ]:
from itertools import count

# count — бесконечная последовательность, можно использовать с любыми числами
for x in count(0.5, 0.5):
    print(x)
    if x >= 2:
        break


# range — конечная последовательность целых чисел
for i in range(1, 5, 1):
    print(i)

`itertools.cycle(iterable)` — это функция, возвращающая итератор, который бесконечно повторяет элементы переданной ему последовательности. После того, как все элементы iterable будут возвращены, перебор начинается сначала, и так далее без остановки.

In [ ]:
colors: list[str] = ["red", "green", "blue"]
# cycle бесконечен, поэтому берём начало через islice
print(list(islice(cycle(colors), 7)))

`itertools.repeat(object, times=None)` — это функция, возвращающая итератор, который выдаёт указанный объект столько раз, сколько задано в параметре `times`. Если параметр `times` не указан, итератор будет бесконечно возвращать объект.

In [ ]:
print(list(repeat("ABC", 5)))  # ровно 5 раз
print(list(islice(repeat(42), 4)))  # times не задан - бесконечно, режем islice

#### permutations, combinations, combinations_with_replacement

In [ ]:
from itertools import permutations, combinations, combinations_with_replacement

`itertools.permutations(iterable, r)` возвращает все возможные перестановки длины `r` из элементов `iterable`. Порядок элементов важен, повторения не допускаются. Если `r` не указано, по умолчанию равно длине последовательности.

In [ ]:
print(list(permutations([1, 2, 3], r=3)))

`itertools.combinations(iterable, r)` создаёт все сочетания длины `r` из элементов `iterable`. Порядок внутри сочетания не важен, и каждый элемент берётся не более одного раза.

Важная тонкость: выбираются **позиции**, а не значения. Если в исходной последовательности есть одинаковые элементы, среди результатов появятся одинаковые сочетания — что и видно в примере ниже на строке `"ABACD"` с двумя буквами `A`.

In [ ]:
# в "ABACD" две буквы A, поэтому ("A", "C", "D") встретится дважды:
print(list(combinations("ABACD", r=3)))

`itertools.combinations_with_replacement(iterable, r)` возвращает все возможные комбинации длины `r` с повторениями из элементов `iterable`. Каждый элемент может входить в комбинацию несколько раз.

In [ ]:
print(list(combinations_with_replacement("ABC", r=2)))